In [1]:
using BenchmarkTools

In [14]:
struct Spectrum
    x::Float64
    y::Float64
    z::Float64
end

struct SurfaceInteraction
    x::Float64
    y::Float64
end

const TextureType = Union{Float64, Spectrum}
abstract type AbstractTexture end
struct ConstantTexture{T <: TextureType} <: AbstractTexture
    value::T
end
function (c::ConstantTexture{T})(si::SurfaceInteraction)::T where T <: TextureType
    return c.value
end
abstract type AbstractMaterial end
struct MatteMaterial <: AbstractMaterial
    Kd::AbstractTexture  # really this should be spectral
    sigma::AbstractTexture  # really this should be float
end

In [15]:
m1 = MatteMaterial(ConstantTexture(1.0), ConstantTexture(Spectrum(0.5, 0.5, 0.5)))
si = SurfaceInteraction(10.0, 20.0)

SurfaceInteraction(10.0, 20.0)

In [16]:
@code_warntype MatteMaterial(ConstantTexture(1.0), ConstantTexture(Spectrum(0.5, 0.5, 0.5)))

MethodInstance for MatteMaterial(::ConstantTexture{Float64}, ::ConstantTexture{Spectrum})
  from MatteMaterial(Kd::AbstractTexture, sigma::AbstractTexture) @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:22
Arguments
  #ctor-self#::Core.Const(MatteMaterial)
  Kd::ConstantTexture{Float64}
  sigma::ConstantTexture{Spectrum}
Body::MatteMaterial
1 ─ %1 = %new(Main.MatteMaterial, Kd, sigma)::Core.PartialStruct(MatteMaterial, Any[ConstantTexture{Float64}, ConstantTexture{Spectrum}])
└──      return %1



In [20]:
@code_warntype m1.Kd(si)

MethodInstance for (::ConstantTexture{Float64})(::SurfaceInteraction)
  from (c::ConstantTexture{T})(si::SurfaceInteraction) where T<:TextureType @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:17
Static Parameters
  T = Float64
Arguments
  c::ConstantTexture{Float64}
  si::SurfaceInteraction
Locals
  @_3::Float64
Body::Float64
1 ─ %1 = $(Expr(:static_parameter, 1))::Core.Const(Float64)
│   %2 = Base.getproperty(c, :value)::Float64
│        (@_3 = %2)
│   %4 = (@_3 isa %1)::Core.Const(true)
└──      goto #3 if not %4
2 ─      goto #4
3 ─      Core.Const(:(Base.convert(%1, @_3)))
└──      Core.Const(:(@_3 = Core.typeassert(%7, %1)))
4 ┄      return @_3



In [22]:
@btime m1.Kd(si)

  35.247 ns (1 allocation: 16 bytes)


1.0

In [21]:
@code_warntype m1.sigma(si)

MethodInstance for (::ConstantTexture{Spectrum})(::SurfaceInteraction)
  from (c::ConstantTexture{T})(si::SurfaceInteraction) where T<:TextureType @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:17
Static Parameters
  T = Spectrum
Arguments
  c::ConstantTexture{Spectrum}
  si::SurfaceInteraction
Locals
  @_3::Spectrum
Body::Spectrum
1 ─ %1 = $(Expr(:static_parameter, 1))::Core.Const(Spectrum)
│   %2 = Base.getproperty(c, :value)::Spectrum
│        (@_3 = %2)
│   %4 = (@_3 isa %1)::Core.Const(true)
└──      goto #3 if not %4
2 ─      goto #4
3 ─      Core.Const(:(Base.convert(%1, @_3)))
└──      Core.Const(:(@_3 = Core.typeassert(%7, %1)))
4 ┄      return @_3



In [23]:
@btime m1.sigma(si)

  39.557 ns (1 allocation: 32 bytes)


Spectrum(0.5, 0.5, 0.5)

In [11]:
@btime m = MatteMaterial(ConstantTexture(1.0), ConstantTexture(Spectrum(0.5, 0.5, 0.5)))

  1.500 ns (0 allocations: 0 bytes)


MatteMaterial(ConstantTexture{Float64}(1.0), ConstantTexture{Spectrum}(Spectrum(0.5, 0.5, 0.5)))

In [30]:
struct Spectrum
    x::Float64
    y::Float64
    z::Float64
end

abstract type AbstractMaterial end
abstract type Texture end
abstract type FloatTexture <: Texture end
abstract type SpectrumTexture <: Texture end
struct ConstantFloatTexture <: FloatTexture
    value::Float64
end
struct ConstantSpectrumTexture <: SpectrumTexture
    value::Spectrum
end
struct MatteMaterial2{S <: SpectrumTexture, F <: FloatTexture} <: AbstractMaterial
    Kd::S
    sigma::F
end

function (c::ConstantFloatTexture)(si::SurfaceInteraction)
    return c.value
end
function (c::ConstantSpectrumTexture)(si::SurfaceInteraction)
    return c.value
end


In [31]:
m2 = MatteMaterial2(ConstantSpectrumTexture(Spectrum(0.5, 0.5, 0.5)), ConstantFloatTexture(1.0))

MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture}(ConstantSpectrumTexture(Spectrum(0.5, 0.5, 0.5)), ConstantFloatTexture(1.0))

In [32]:
@code_warntype MatteMaterial2(ConstantSpectrumTexture(Spectrum(0.5, 0.5, 0.5)), ConstantFloatTexture(1.0))

MethodInstance for MatteMaterial2(::ConstantSpectrumTexture, ::ConstantFloatTexture)
  from MatteMaterial2(Kd::S, sigma::F) where {S<:SpectrumTexture, F<:FloatTexture} @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:18
Static Parameters
  S = ConstantSpectrumTexture
  F = ConstantFloatTexture
Arguments
  #self#::Type{MatteMaterial2}
  Kd::ConstantSpectrumTexture
  sigma::ConstantFloatTexture
Body::MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture}
1 ─ %1 = Core.apply_type(Main.MatteMaterial2, $(Expr(:static_parameter, 1)), $(Expr(:static_parameter, 2)))::Core.Const(MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture})
│   %2 = %new(%1, Kd, sigma)::MatteMaterial2{ConstantSpectrumTexture, ConstantFloatTexture}
└──      return %2



In [33]:
@btime m2.Kd(si)

  68.239 ns (3 allocations: 112 bytes)


Spectrum(0.5, 0.5, 0.5)

In [35]:
@code_warntype m2.Kd(si)

MethodInstance for (::ConstantSpectrumTexture)(::SurfaceInteraction)
  from (c::ConstantSpectrumTexture)(si::SurfaceInteraction) @ Main ~/random_stuff/PBRJ/scratch/texture_material_parameterization/play.ipynb:25
Arguments
  c::ConstantSpectrumTexture
  si::SurfaceInteraction
Body::Spectrum
1 ─ %1 = Base.getproperty(c, :value)::Spectrum
└──      return %1



In [34]:
@btime m2.sigma(si)

  68.750 ns (3 allocations: 80 bytes)


1.0